<a href="https://colab.research.google.com/github/TerteryanTatev/Optimization-Methods/blob/main/Simplex-Method-Solver/simplex_method_solver.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd


# x3 >= 30
SHIFT = 30


cols = ["x1","x2","x3'","x4","y1","y2","y3",
       "s1","s2","s3","s4","s5","s6","s7"]


Cj = np.array([5500,4800,3200,8500,-1000,-1000,-1000,0,0,0,0,0,0,0])


A = np.array([
   [2.5,1.8,1.2,4.5,-1,0,0,1,0,0,0,0,0,0],
   [4,3,2,6,0,-1,0,0,1,0,0,0,0,0],
   [1.5,2.5,1,3,0,0,-1,0,0,1,0,0,0,0],
   [0,0,0,0,1,0,0,0,0,0,1,0,0,0],
   [0,0,0,0,0,1,0,0,0,0,0,1,0,0],
   [0,0,0,0,0,0,1,0,0,0,0,0,1,0],
   [0,0,0,1,0,0,0,0,0,0,0,0,0,1]
])


b = np.array([
   400 - 1.2*SHIFT,
   700 - 2*SHIFT,
   350 - 1*SHIFT,
   100,
   150,
   100,
   50
])


def print_model():
   print("\nՍԿԶԲՆԱԿԱՆ ՄՈԴԵԼ\n")


   print("Maximize:")
   print("Z = 5500x1 + 4800x2 + 3200x3 + 8500x4 - 1000(y1 + y2 + y3)\n")


   print("Subject to:")
   print("2.5x1 + 1.8x2 + 1.2x3 + 4.5x4 <= 400 + y1")
   print("4x1 + 3x2 + 2x3 + 6x4 <= 700 + y2")
   print("1.5x1 + 2.5x2 + 1x3 + 3x4 <= 350 + y3")
   print("y1 <= 100, y2 <= 150, y3 <= 100")
   print("x4 <= 50")
   print("x3 >= 30")
   print("x1, x2, x3, x4, y1, y2, y3 >= 0")


def print_canonical():
   print("\nԿԱՆՈՆԱԿԱՆ ՏԵՍՔ\n")


   print("Փոխարինում ենք  x3 = x3' + 30\n")


   print("2.5x1 + 1.8x2 + 1.2x3' + 4.5x4 - y1 + s1 = 364")
   print("4x1 + 3x2 + 2x3' + 6x4 - y2 + s2 = 640")
   print("1.5x1 + 2.5x2 + 1x3' + 3x4 - y3 + s3 = 320")


   print("y1 + s4 = 100")
   print("y2 + s5 = 150")
   print("y3 + s6 = 100")
   print("x4 + s7 = 50")


   print("\nZ  = 5500x1 + 4800x2 + 3200x3' + 8500x4 - 1000y1 - 1000y2 - 1000y3 -96000")


basis = ["s1","s2","s3","s4","s5","s6","s7"]
Cb = np.zeros(len(basis))


step = 0


def show_table(A, b, basis, Cb, step, entering=None, leaving=None):
   Zj = Cb @ A
   delta = Zj - Cj


   df = pd.DataFrame(A, columns=cols)
   df["b"] = b
   df.insert(0, "Cb", Cb)
   df.insert(0, "Basis", basis)

   delta_row_values = [np.nan, np.nan] + list(np.round(delta, 2)) + [np.nan]
   delta_row_df = pd.DataFrame([delta_row_values], columns=df.columns, index=['Zj-Cj'])


   df = pd.concat([df, delta_row_df])


   print(f"\nITERATION {step} ")


   if entering:
       print(f"մտնում է: {entering}")
   if leaving:
       print(f" դուրս է գալիս: {leaving}")


   styled = df.style.set_properties(**{
       "background-color": "white",
       "border": "1px solid black",
       "text-align": "center",
       'color':'black'
   })


   display(styled)


print_model()
print_canonical()


while True:
   step += 1


   Zj = Cb @ A
   delta = Zj - Cj


   if all(delta >= 0):
       show_table(A, b, basis, Cb, step)
       print("\n OPTIMAL")
       break

   entering_idx = np.argmin(delta)
   entering = cols[entering_idx]


   ratios = []
   for i in range(len(b)):
       if A[i, entering_idx] > 0:
           ratios.append(b[i] / A[i, entering_idx])
       else:
           ratios.append(np.inf)


   leaving_idx = np.argmin(ratios)
   leaving = basis[leaving_idx]


   show_table(A, b, basis, Cb, step, entering, leaving)


   pivot = A[leaving_idx, entering_idx]


   A[leaving_idx] /= pivot
   b[leaving_idx] /= pivot


   for i in range(len(A)):
       if i != leaving_idx:
           factor = A[i, entering_idx]
           A[i] -= factor * A[leaving_idx]
           b[i] -= factor * b[leaving_idx]


   basis[leaving_idx] = entering
   Cb[leaving_idx] = Cj[entering_idx]




solution = {var: 0 for var in cols}


for i, var in enumerate(basis):
   solution[var] = b[i]


x3 = solution["x3'"] + SHIFT


Z = (5500*solution["x1"] + 4800*solution["x2"] +
    3200*x3 + 8500*solution["x4"] -
    1000*(solution["y1"] + solution["y2"] + solution["y3"]))


print("\n════════════════════════════════")
print("OPTIMAL SOLUTION")
print("════════════════════════════════")
print(f"x1  = {solution['x1']:.2f}")
print(f"x2  = {solution['x2']:.2f}")
print(f"x3  = {x3:.2f}  (x3' = {solution['x3\'']:.2f} + {SHIFT})")
print(f"x4  = {solution['x4']:.2f}")
print(f"y1  = {solution['y1']:.2f}")
print(f"y2  = {solution['y2']:.2f}")
print(f"y3  = {solution['y3']:.2f}")
print(f"\nZ*  = {Z:,.0f}")




ՍԿԶԲՆԱԿԱՆ ՄՈԴԵԼ

Maximize:
Z = 5500x1 + 4800x2 + 3200x3 + 8500x4 - 1000(y1 + y2 + y3)

Subject to:
2.5x1 + 1.8x2 + 1.2x3 + 4.5x4 <= 400 + y1
4x1 + 3x2 + 2x3 + 6x4 <= 700 + y2
1.5x1 + 2.5x2 + 1x3 + 3x4 <= 350 + y3
y1 <= 100, y2 <= 150, y3 <= 100
x4 <= 50
x3 >= 30
x1, x2, x3, x4, y1, y2, y3 >= 0

ԿԱՆՈՆԱԿԱՆ ՏԵՍՔ

Փոխարինում ենք  x3 = x3' + 30

2.5x1 + 1.8x2 + 1.2x3' + 4.5x4 - y1 + s1 = 364
4x1 + 3x2 + 2x3' + 6x4 - y2 + s2 = 640
1.5x1 + 2.5x2 + 1x3' + 3x4 - y3 + s3 = 320
y1 + s4 = 100
y2 + s5 = 150
y3 + s6 = 100
x4 + s7 = 50

Z  = 5500x1 + 4800x2 + 3200x3' + 8500x4 - 1000y1 - 1000y2 - 1000y3 -96000

ITERATION 1 
մտնում է: x4
 դուրս է գալիս: s7


,Basis,Cb,x1,x2,x3',x4,y1,y2,y3,s1,s2,s3,s4,s5,s6,s7,b
0,s1,0.000000,2.500000,1.800000,1.200000,4.500000,-1.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,364.000000
1,s2,0.000000,4.000000,3.000000,2.000000,6.000000,0.000000,-1.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,640.000000
2,s3,0.000000,1.500000,2.500000,1.000000,3.000000,0.000000,0.000000,-1.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,320.000000
3,s4,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,100.000000
4,s5,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,150.000000
5,s6,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,100.000000
6,s7,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,50.000000
Zj-Cj,nan,nan,-5500.000000,-4800.000000,-3200.000000,-8500.000000,1000.000000,1000.000000,1000.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,nan



ITERATION 2 
մտնում է: x1
 դուրս է գալիս: s1


,Basis,Cb,x1,x2,x3',x4,y1,y2,y3,s1,s2,s3,s4,s5,s6,s7,b
0,s1,0.000000,2.500000,1.800000,1.200000,0.000000,-1.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,-4.500000,139.000000
1,s2,0.000000,4.000000,3.000000,2.000000,0.000000,0.000000,-1.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,-6.000000,340.000000
2,s3,0.000000,1.500000,2.500000,1.000000,0.000000,0.000000,0.000000,-1.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,-3.000000,170.000000
3,s4,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,100.000000
4,s5,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,150.000000
5,s6,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,100.000000
6,x4,8500.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,50.000000
Zj-Cj,nan,nan,-5500.000000,-4800.000000,-3200.000000,0.000000,1000.000000,1000.000000,1000.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,8500.000000,nan



ITERATION 3 
մտնում է: s7
 դուրս է գալիս: x4


...


ITERATION 7 
մտնում է: y3
 դուրս է գալիս: x1


,Basis,Cb,x1,x2,x3',x4,y1,y2,y3,s1,s2,s3,s4,s5,s6,s7,b
0,x1,5500.000000,1.000000,-2.000000,0.000000,0.000000,0.000000,-1.000000,2.000000,0.000000,1.000000,-2.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1,y1,-1000.000000,0.000000,-0.200000,0.000000,-0.900000,1.000000,-0.700000,0.200000,-1.000000,0.700000,-0.200000,0.000000,0.000000,0.000000,0.000000,20.000000
2,x3',3200.000000,0.000000,5.500000,1.000000,3.000000,0.000000,1.500000,-4.000000,0.000000,-1.500000,4.000000,0.000000,0.000000,0.000000,0.000000,320.000000
3,s4,0.000000,0.000000,0.200000,0.000000,0.900000,0.000000,0.700000,-0.200000,1.000000,-0.700000,0.200000,1.000000,0.000000,0.000000,0.000000,80.000000
4,s5,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,150.000000
5,s6,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,100.000000
6,s7,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,50.000000
Zj-Cj,nan,nan,0.000000,2000.000000,0.000000,2000.000000,0.000000,1000.000000,-1000.000000,1000.000000,-0.000000,2000.000000,0.000000,0.000000,0.000000,0.000000,nan



ITERATION 8 


,Basis,Cb,x1,x2,x3',x4,y1,y2,y3,s1,s2,s3,s4,s5,s6,s7,b
0,y3,-1000.000000,0.500000,-1.000000,0.000000,0.000000,0.000000,-0.500000,1.000000,0.000000,0.500000,-1.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1,y1,-1000.000000,-0.100000,0.000000,0.000000,-0.900000,1.000000,-0.600000,0.000000,-1.000000,0.600000,0.000000,0.000000,0.000000,0.000000,0.000000,20.000000
2,x3',3200.000000,2.000000,1.500000,1.000000,3.000000,0.000000,-0.500000,0.000000,0.000000,0.500000,0.000000,0.000000,0.000000,0.000000,0.000000,320.000000
3,s4,0.000000,0.100000,-0.000000,0.000000,0.900000,0.000000,0.600000,0.000000,1.000000,-0.600000,0.000000,1.000000,0.000000,0.000000,0.000000,80.000000
4,s5,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,150.000000
5,s6,0.000000,-0.500000,1.000000,0.000000,0.000000,0.000000,0.500000,0.000000,0.000000,-0.500000,1.000000,0.000000,0.000000,1.000000,0.000000,100.000000
6,s7,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,50.000000
Zj-Cj,nan,nan,500.000000,1000.000000,0.000000,2000.000000,0.000000,500.000000,0.000000,1000.000000,500.000000,1000.000000,0.000000,0.000000,0.000000,0.000000,nan



 OPTIMAL

════════════════════════════════
OPTIMAL SOLUTION
════════════════════════════════
x1  = 0.00
x2  = 0.00
x3  = 350.00  (x3' = 320.00 + 30)
x4  = 0.00
y1  = 20.00
y2  = 0.00
y3  = 0.00

Z*  = 1,100,000


In [ ]:

print("\nSOLUTION")
print("x1 =", solution["x1"])
print("x2 =", solution["x2"])
print("x3 =", x3)
print("x4 =", solution["x4"])
print("y1 =", solution["y1"])
print("y2 =", solution["y2"])
print("y3 =", solution["y3"])

Z = sum(Cj[i]*solution[cols[i]] for i in range(len(cols))) + 3200*SHIFT
print("\nZ =", Z)


SOLUTION
x1 = 0
x2 = 0
x3 = 350.0
x4 = 0
y1 = 19.999999999999982
y2 = 0
y3 = 1.4210854715202004e-14

Z = 1100000.0



ՍԿԶԲՆԱԿԱՆ ՄՈԴԵԼ

Maximize:
Z = 5500x1 + 4800x2 + 3200x3 + 8500x4 - 1000(y1 + y2 + y3)

Subject to:
2.5x1 + 1.8x2 + 1.2x3 + 4.5x4 <= 400 + y1
4x1 + 3x2 + 2x3 + 6x4 <= 700 + y2
1.5x1 + 2.5x2 + 1x3 + 3x4 <= 350 + y3
y1 <= 100, y2 <= 150, y3 <= 100
x4 <= 50
x3 >= 30
x1, x2, x3, x4, y1, y2, y3 >= 0

ԿԱՆՈՆԱԿԱՆ ՏԵՍՔ

Փոխարինում ենք  x3 = x3' + 30

2.5x1 + 1.8x2 + 1.2x3' + 4.5x4 - y1 + s1 = 364
4x1 + 3x2 + 2x3' + 6x4 - y2 + s2 = 640
1.5x1 + 2.5x2 + 1x3' + 3x4 - y3 + s3 = 320
y1 + s4 = 100
y2 + s5 = 150
y3 + s6 = 100
x4 + s7 = 50

Z  = 5500x1 + 4800x2 + 3200x3' + 8500x4 - 1000y1 - 1000y2 - 1000y3 -96000

ITERATION 1 
մտնում է: x4
 դուրս է գալիս: s7


,Basis,Cb,x1,x2,x3',x4,y1,y2,y3,s1,s2,s3,s4,s5,s6,s7,b
0,s1,0.000000,2.500000,1.800000,1.200000,4.500000,-1.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,364.000000
1,s2,0.000000,4.000000,3.000000,2.000000,6.000000,0.000000,-1.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,640.000000
2,s3,0.000000,1.500000,2.500000,1.000000,3.000000,0.000000,0.000000,-1.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,320.000000
3,s4,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,100.000000
4,s5,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,150.000000
5,s6,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,100.000000
6,s7,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,50.000000
Zj-Cj,nan,nan,-5500.000000,-4800.000000,-3200.000000,-8500.000000,1000.000000,1000.000000,1000.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,nan



ITERATION 2 
մտնում է: x1
 դուրս է գալիս: s1


,Basis,Cb,x1,x2,x3',x4,y1,y2,y3,s1,s2,s3,s4,s5,s6,s7,b
0,s1,0.000000,2.500000,1.800000,1.200000,0.000000,-1.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,-4.500000,139.000000
1,s2,0.000000,4.000000,3.000000,2.000000,0.000000,0.000000,-1.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,-6.000000,340.000000
2,s3,0.000000,1.500000,2.500000,1.000000,0.000000,0.000000,0.000000,-1.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,-3.000000,170.000000
3,s4,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,100.000000
4,s5,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,150.000000
5,s6,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,100.000000
6,x4,8500.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,50.000000
Zj-Cj,nan,nan,-5500.000000,-4800.000000,-3200.000000,0.000000,1000.000000,1000.000000,1000.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,8500.000000,nan



ITERATION 3 
մտնում է: s7
 դուրս է գալիս: x4


...


ITERATION 7 
մտնում է: y3
 դուրս է գալիս: x1


,Basis,Cb,x1,x2,x3',x4,y1,y2,y3,s1,s2,s3,s4,s5,s6,s7,b
0,x1,5500.000000,1.000000,-2.000000,0.000000,0.000000,0.000000,-1.000000,2.000000,0.000000,1.000000,-2.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1,y1,-1000.000000,0.000000,-0.200000,0.000000,-0.900000,1.000000,-0.700000,0.200000,-1.000000,0.700000,-0.200000,0.000000,0.000000,0.000000,0.000000,20.000000
2,x3',3200.000000,0.000000,5.500000,1.000000,3.000000,0.000000,1.500000,-4.000000,0.000000,-1.500000,4.000000,0.000000,0.000000,0.000000,0.000000,320.000000
3,s4,0.000000,0.000000,0.200000,0.000000,0.900000,0.000000,0.700000,-0.200000,1.000000,-0.700000,0.200000,1.000000,0.000000,0.000000,0.000000,80.000000
4,s5,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,150.000000
5,s6,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,100.000000
6,s7,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,50.000000
Zj-Cj,nan,nan,0.000000,2000.000000,0.000000,2000.000000,0.000000,1000.000000,-1000.000000,1000.000000,-0.000000,2000.000000,0.000000,0.000000,0.000000,0.000000,nan



ITERATION 8 


,Basis,Cb,x1,x2,x3',x4,y1,y2,y3,s1,s2,s3,s4,s5,s6,s7,b
0,y3,-1000.000000,0.500000,-1.000000,0.000000,0.000000,0.000000,-0.500000,1.000000,0.000000,0.500000,-1.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1,y1,-1000.000000,-0.100000,0.000000,0.000000,-0.900000,1.000000,-0.600000,0.000000,-1.000000,0.600000,0.000000,0.000000,0.000000,0.000000,0.000000,20.000000
2,x3',3200.000000,2.000000,1.500000,1.000000,3.000000,0.000000,-0.500000,0.000000,0.000000,0.500000,0.000000,0.000000,0.000000,0.000000,0.000000,320.000000
3,s4,0.000000,0.100000,-0.000000,0.000000,0.900000,0.000000,0.600000,0.000000,1.000000,-0.600000,0.000000,1.000000,0.000000,0.000000,0.000000,80.000000
4,s5,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,150.000000
5,s6,0.000000,-0.500000,1.000000,0.000000,0.000000,0.000000,0.500000,0.000000,0.000000,-0.500000,1.000000,0.000000,0.000000,1.000000,0.000000,100.000000
6,s7,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,50.000000
Zj-Cj,nan,nan,500.000000,1000.000000,0.000000,2000.000000,0.000000,500.000000,0.000000,1000.000000,500.000000,1000.000000,0.000000,0.000000,0.000000,0.000000,nan



 OPTIMAL

════════════════════════════════
OPTIMAL SOLUTION
════════════════════════════════
x1  = 0.00
x2  = 0.00
x3  = 350.00  (x3' = 320.00 + 30)
x4  = 0.00
y1  = 20.00
y2  = 0.00
y3  = 0.00

Z*  = 1,100,000
